In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [23]:
df = pd.read_csv("../data/raw/Bank-Customer-Churn.csv")

In [24]:
# Quitar espacios en blanco de los nombres de las columnas
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

,rownumber,customerid,surname,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,complain,satisfaction_score,card_type,point_earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [25]:
# Eliminar columnas que no sirven para nada
df = df.drop(columns=["rownumber", "customerid", "surname"])
df = df.drop(columns=["complain"]) # no se aun ya que predice el churn, si todo va mal lo meto :/
df.head()

,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,satisfaction_score,card_type,point_earned
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,2,DIAMOND,464
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,3,DIAMOND,456
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,3,DIAMOND,377
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,5,GOLD,350
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,5,GOLD,425


In [26]:
# Encoding de las variables categoricas
df["gender"] = df["gender"].map({
    "Male": 0,
    "Female": 1
})

df = pd.get_dummies(df, columns=["geography","card_type"], drop_first=True) # sin drop_first porque lo quiero más legible y luego se me olvida lo que hace
# al final lo pongo a True porque según he leido es mas claro y rinde mejor

In [27]:
df.head()

,creditscore,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,satisfaction_score,point_earned,geography_Germany,geography_Spain,card_type_GOLD,card_type_PLATINUM,card_type_SILVER
0,619,1,42,2,0.00,1,1,1,101348.88,1,2,464,False,False,False,False,False
1,608,1,41,1,83807.86,1,0,1,112542.58,0,3,456,False,True,False,False,False
2,502,1,42,8,159660.80,3,1,0,113931.57,1,3,377,False,False,False,False,False
3,699,1,39,1,0.00,2,0,0,93826.63,0,5,350,False,False,True,False,False
4,850,1,43,2,125510.82,1,1,1,79084.10,0,5,425,False,True,True,False,False


In [28]:
# Booleanos a int
bool_cols = df.select_dtypes(include="bool").columns

df[bool_cols] = df[bool_cols].astype(int)
df.head()

,creditscore,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,satisfaction_score,point_earned,geography_Germany,geography_Spain,card_type_GOLD,card_type_PLATINUM,card_type_SILVER
0,619,1,42,2,0.00,1,1,1,101348.88,1,2,464,0,0,0,0,0
1,608,1,41,1,83807.86,1,0,1,112542.58,0,3,456,0,1,0,0,0
2,502,1,42,8,159660.80,3,1,0,113931.57,1,3,377,0,0,0,0,0
3,699,1,39,1,0.00,2,0,0,93826.63,0,5,350,0,0,1,0,0
4,850,1,43,2,125510.82,1,1,1,79084.10,0,5,425,0,1,1,0,0


In [29]:
# guardamos el df preparado para train test
df.to_csv("../data/processed/churn_processed.csv", index=False)

# TRAIN TEST

In [30]:
# separar columnas target
X = df.drop("exited", axis=1)
y = df["exited"]

In [31]:
# ver el desbalance
print(y.value_counts(normalize=True) * 100)

exited
0    79.62
1    20.38
Name: proportion, dtype: float64


In [32]:
# pasamos a hacer traintest
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # stratify para mantener la misma proporción de clases en train y test
)

### Modelos que NECESITAN escalado
- Logistic Regression
- SVM
- KNN
Redes neuronales
### Modelos que NO lo necesitan
- Decision Tree
- Random Forest
- XGBoost
- LightGBM

Guardar version escalada y sin escalar

In [34]:
numerical_cols = [
    "creditscore",
    "age",
    "tenure",
    "balance",
    "numofproducts",
    "estimatedsalary",
    "satisfaction_score",
    "point_earned"
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols]
)

X_test_scaled[numerical_cols] = scaler.transform(
    X_test[numerical_cols]
)

In [35]:
# Guardar escalados y sin escalar
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

X_train_scaled.to_csv("../data/processed/X_train_scaled.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test_scaled.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)